In [1]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# 2. Definir la ruta principal de tu Drive
drive_path = '/content/drive/MyDrive'

# 3. Crear la carpeta 'ppe' (si ya existe, no dará error gracias a exist_ok=True)
ppe_path = os.path.join(drive_path, 'ppe')
os.makedirs(ppe_path, exist_ok=True)

# 4. Establecer la carpeta 'ppe' como el directorio por defecto (working directory)
os.chdir(ppe_path)

# 5. Comprobar que en efecto ahora estás dentro de esa carpeta
print(f"Directorio actual por defecto: {os.getcwd()}")


Mounted at /content/drive
Directorio actual por defecto: /content/drive/MyDrive/ppe


In [ ]:
!pip install ultralytics roboflow --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 110.1 MB/s eta 0:00:0000:01


: 

In [3]:
import os
import shutil
from roboflow import Roboflow

# Usando tu carpeta por default que configuramos antes
ruta_descarga = '/content/drive/MyDrive/ppe'

# Verificamos si la carpeta ya existe
if os.path.exists(ruta_descarga):
    print(f"Borrando la carpeta existente y su contenido en {ruta_descarga}...")
    shutil.rmtree(ruta_descarga) # Esto borra la carpeta y todo lo que tenga adentro
    print("¡Limpieza completada!")

print("Iniciando la descarga del dataset...")

rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")
version = project.version(1)

# Agregamos location=ruta_descarga para obligarlo a descargarse ahí
dataset = version.download("yolov8", location=ruta_descarga)

print("¡Descarga completada exitosamente en la carpeta ppe!")


Borrando la carpeta existente y su contenido en /content/drive/MyDrive/ppe...
¡Limpieza completada!
Iniciando la descarga del dataset...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/ppe in yolov8:: 100%|██████████| 21608/21608 [03:28<00:00, 103.45it/s]


: 

: 

In [ ]:
# Ver el contenido general de la carpeta ppe
!ls -lh /content/drive/MyDrive/ppe


In [ ]:
!cat /content/drive/MyDrive/ppe/data.yaml


In [ ]:
# Esto nos dirá la ruta exacta donde Roboflow guardó el archivo data.yaml y las imágenes
print(dataset.location)


In [ ]:
import os

ruta = '/content/drive/MyDrive/ppe'

# 1. Le pedimos a Python directamente que nos diga qué hay, a veces bash (!ls) falla
archivos_en_carpeta = os.listdir(ruta)
print(f"Archivos encontrados por Python: {archivos_en_carpeta}")

# 2. Si sigue vacío, forzamos a Roboflow a descargarlo de nuevo
if len(archivos_en_carpeta) == 0:
    print("\nLa carpeta está vacía. Forzando a Roboflow a descargar de nuevo...")
    # Asegurarnos de estar parados en la carpeta correcta
    os.chdir(ruta)
    dataset = version.download("yolov8")
    print("\nDescarga reintentada.")
    print(f"Archivos ahora: {os.listdir(ruta)}")


In [ ]:
!cat /content/drive/MyDrive/ppe/ppe-factory-1/data.yaml

In [ ]:
from ultralytics import YOLO

# 1. Cargar el modelo YOLOv8 en su versión 'nano'
model = YOLO('yolov8n.pt')

# 2. La ruta exacta que nos confirmaste
ruta_al_yaml = '/content/drive/MyDrive/ppe/ppe-factory-1/data.yaml'

# 3. La carpeta donde quieres que queden guardados los resultados del entrenamiento 
# (Los guardaremos directo en tu carpeta ppe)
carpeta_resultados = '/content/drive/MyDrive/ppe/resultados_entrenamiento'

# 4. Entrenar el modelo
resultados = model.train(
    data=ruta_al_yaml, 
    epochs=5, # Puedes subirlo a 50 o 100 si quieres un modelo más preciso
    imgsz=640,
    project=carpeta_resultados, # Guarda toda la información en nuestro Drive
    name='mi_primer_entrenamiento' # Nombre final de la carpeta: ppe/resultados_entrenamiento/mi_primer_entrenamiento
)

print("\n¡Entrenamiento finalizado! Tus resultados están a salvo en Google Drive.")
